In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import xarray as xr

def read_sbe_cnv_to_xarray(file):
    file = Path(file)

    header = []
    data_lines = []
    in_data = False

    with open(file, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if line == "*END*":
                in_data = True
                continue

            if in_data:
                if line:
                    data_lines.append(line)
            else:
                header.append(line)

    # -----------------------
    # Parse column names
    # -----------------------
    names = {}
    for line in header:
        m = re.match(r"# name\s+(\d+)\s+=\s+([^:]+):", line)
        if m:
            idx = int(m.group(1))
            varname = m.group(2).strip()
            names[idx] = varname

    columns = [names[i] for i in sorted(names)]

    # -----------------------
    # Parse metadata
    # -----------------------
    attrs = {}

    for line in header:
        if line.startswith("* NMEA Latitude"):
            attrs["NMEA Latitude"] = line.split("=")[1].strip()
        elif line.startswith("* NMEA Longitude"):
            attrs["NMEA Longitude"] = line.split("=")[1].strip()
        elif line.startswith("# start_time"):
            attrs["start_time"] = line.split("=")[1].split("[")[0].strip()
        elif line.startswith("# bad_flag"):
            attrs["bad_flag"] = float(line.split("=")[1].strip())

    # -----------------------
    # Read numeric data
    # -----------------------
    from io import StringIO

    df = pd.read_csv(
        StringIO("\n".join(data_lines)),
        sep=r"\s+",
        names=columns,
        engine="python"
    )

    # Replace bad values
    if "bad_flag" in attrs:
        df = df.replace(attrs["bad_flag"], np.nan)

    # -----------------------
    # Rename useful variables
    # -----------------------
    rename = {
        "prDM": "press",
        "depSM": "depth",
        "latitude": "lat",
        "longitude": "lon",
        "t090C": "temp",
        "t190C": "temp2",
        "sal00": "sal",
        "potemp090C": "theta",
        "sigma-t00": "sigma_t",
        "sbeox0ML/L": "O2",
        "timeS": "time_elapsed",
    }

    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    # Sort by pressure, just in case
    df = df.sort_values("press")

    # -----------------------
    # Make xarray Dataset
    # -----------------------
    ds = df.set_index("press").to_xarray()

    # Make lat/lon scalar coordinates if they are basically constant
    if "lat" in ds:
        lat_mean = float(ds["lat"].mean())
        ds = ds.drop_vars("lat")
        ds = ds.assign_coords(lat=lat_mean)

    if "lon" in ds:
        lon_mean = float(ds["lon"].mean())
        ds = ds.drop_vars("lon")
        ds = ds.assign_coords(lon=lon_mean)

    if "start_time" in attrs:
        ds = ds.assign_coords(time=pd.to_datetime(attrs["start_time"]))

    ds.attrs.update(attrs)

    return ds

In [ ]:
from pathlib import Path

folder = Path("//thepenguin/penguin2/Data/TNB/mooring/LDEO/2017_18 Araon Cruise (ANA08C)/Data/Cruise/ANA08C/KOPRI/CTD/2017_ANA08C_TNB/CTD_processed/07_binaveraged")

In [ ]:
sorted(folder.glob("*.cnv"))

In [ ]:
ds = read_sbe_cnv_to_xarray(file="//thepenguin/penguin2/Data/TNB/mooring/LDEO/2017_18 Araon Cruise (ANA08C)/Data/Cruise/ANA08C/KOPRI/CTD/2017_ANA08C_TNB/CTD_processed/07_binaveraged/ANA08C_TNB_CTD01.cnv")

In [ ]:
ds

In [ ]:
fig = plt.figure()
plt.plot(ds.temp,ds.press)
plt.gca().invert_yaxis()

In [ ]:
fig = plt.figure()
plt.plot(ds.sal,ds.press)
plt.gca().invert_yaxis()

In [ ]:
# load and format all files 
casts = {}

for i, file in enumerate(sorted(folder.glob("*.cnv")), start=1):
    
    ds = read_sbe_cnv_to_xarray(file)
    
    ds.attrs["source_file"] = file.name
    ds = ds.assign_coords(station=i)
    
    casts[i] = ds

In [ ]:
casts

In [ ]:
cast = casts[30]

In [ ]:
cast

In [ ]:
diff = cast.sal - cast.sal11

In [ ]:
fig = plt.figure()
diff.plot()

In [ ]:
diff

In [ ]:
# updated data figure 
fig, ax = plt.subplots()
ax.plot(cast.temp,cast.press,color='tab:red',label='Temp')
ax.invert_yaxis()
ax1 = ax.twiny()
ax1.plot(cast.sal11,cast.press,color='tab:blue',label='salinity')
plt.legend()
ax.set_ylabel('Pressure [dbar]')
ax.set_xlabel('Temp')
ax1.set_xlabel('Sal')

In [ ]:
%matplotlib widget

In [ ]:
cast = casts[20]

In [ ]:
cast

In [ ]:
cast = casts[20]

fig, ax = plt.subplots()
ax.plot(cast.temp,cast.press,color='tab:red',label='Temp')
ax.invert_yaxis()
ax1 = ax.twiny()
ax1.plot(cast.sal,cast.press,color='tab:blue',label='salinity')
plt.legend()
ax.set_ylabel('Pressure [dbar]')
ax.set_xlabel('Temp')
ax1.set_xlabel('Sal')

In [ ]:
import matplotlib.pyplot as plt
import os

# directory to save figures
save_dir = "ctd_profile_TNBD_prelim"
os.makedirs(save_dir, exist_ok=True)

# loop through all casts
for key, cast in casts.items():

    fig, axes = plt.subplots(
        ncols=2,
        figsize=(10, 6),
        constrained_layout=True,
        sharey=False
    )

    # ---------------- FULL PROFILE ----------------
    ax = axes[0]

    ax.plot(cast.temp, cast.press, color='tab:red', label='Temperature')
    ax.invert_yaxis()

    ax_sal = ax.twiny()
    ax_sal.plot(cast.sal, cast.press, color='tab:blue', label='Salinity')

    ax.set_ylabel('Pressure [dbar]')
    ax.set_xlabel('Temperature')
    ax_sal.set_xlabel('Salinity')

    ax.set_title('Full Profile')

    # optional combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax_sal.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

    # ---------------- UPPER 200 DBAR ----------------
    ax2 = axes[1]

    # subset near surface
    cast_surface = cast.where(cast.press <= 200, drop=True)

    ax2.plot(
        cast_surface.temp,
        cast_surface.press,
        color='tab:red',
        label='Temperature'
    )

    ax2.invert_yaxis()

    ax2_sal = ax2.twiny()
    ax2_sal.plot(
        cast_surface.sal,
        cast_surface.press,
        color='tab:blue',
        label='Salinity'
    )

    ax2.set_ylim(200, 0)

    ax2.set_xlabel('Temperature')
    ax2_sal.set_xlabel('Salinity')

    ax2.set_title('Upper 200 dbar')

    # figure title
    fig.suptitle(f'CTD Cast {key}', fontsize=14)

    # save figure
    fig.savefig(
        f"{save_dir}/cast_{key}.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close(fig)

### Plot CTD stations on Map 

In [ ]:
bath = xr.open_dataset("//thepenguin/penguin2/Data/TNB/bathymetry/bath_ross_amundsen.nc")

In [ ]:
import pandas as pd
import geopandas as gpd
import cartopy.crs as ccrs
from shapely.validation import make_valid

# read fresh
gdf = gpd.read_file(
    "/Users/mblan/Documents/research/data/add_coastline_high_res_polygon_v7_11/add_coastline_high_res_polygon_v7_11.shp"
)

if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")
else:
    gdf = gdf.to_crs("EPSG:4326")

# better than buffer(0)
gdf["geometry"] = gdf.geometry.apply(make_valid)

ice = gdf[gdf["surface"].isin(["ice shelf", "ice tongue"])]
land = gdf[gdf["surface"].isin(["land"])]

In [ ]:
def subset_dateline_gdf(gdf, lat_min=-78, lat_max=-60):
    # Ross sector: 160E to 180, plus 180 to 160W
    east = gdf.cx[160:180, lat_min:lat_max]
    west = gdf.cx[-180:-160, lat_min:lat_max]

    out = pd.concat([east, west], ignore_index=True)
    return gpd.GeoDataFrame(out, crs=gdf.crs)

In [ ]:
def TNB_study_site(
    bath,
    ice,
    land,
    lon_min=160,
    lon_max=200,
    lat_min=-78,
    lat_max=-60,
    figsize=(8, 6),
    central_longitude=165,
    contour_levels=None,
    n_contours=5,
    contour_min=200,
    contour_max=1200,
    contour_color="black",
    contour_lw=0.7,
    label_contours=True,
    gridline_lons=None,
    gridline_lats=None,
):
    """
    Create a reusable Terra Nova Bay / Ross Sea study-site map.

    Parameters
    ----------
    bath : xarray.Dataset
        GEBCO bathymetry dataset with variables lon, lat, elevation.
    ice, land : geopandas.GeoDataFrame
        SCAR ADD clipped/unclipped ice and land polygons.
    lon_min, lon_max, lat_min, lat_max : float
        Map extent in lon/lat.
    contour_levels : list or array, optional
        Specific bathymetry contour levels in positive meters.
        Example: [400, 600, 800, 1000]
    n_contours : int
        Number of contours if contour_levels is not provided.
    contour_min, contour_max : float
        Min/max depth contours if contour_levels is not provided.
    """

    # -----------------------------
    # Projection / figure
    # -----------------------------
    proj = ccrs.SouthPolarStereo(central_longitude=central_longitude)

    fig, ax = plt.subplots(
        figsize=figsize,
        subplot_kw={"projection": proj},
        constrained_layout=True
    )

    ax.set_extent(
        [lon_min, lon_max, lat_min, lat_max],
        crs=ccrs.PlateCarree()
    )

    # -----------------------------
    # Gridlines
    # -----------------------------
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.5,
        color="gray",
        alpha=0.5,
        linestyle="--"
    )

    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    if gridline_lons is not None:
        gl.xlocator = mticker.FixedLocator(gridline_lons)

    if gridline_lats is not None:
        gl.ylocator = mticker.FixedLocator(gridline_lats)

    # -----------------------------
    # Bathymetry subset
    # -----------------------------
    bath_sub = bath.sortby("lat").sortby("lon").sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )

    depth = -bath_sub["elevation"]

    if contour_levels is None:
        contour_levels = np.linspace(contour_min, contour_max, n_contours)

    contours = ax.contour(
        depth["lon"],
        depth["lat"],
        depth,
        levels=contour_levels,
        colors=contour_color,
        linewidths=contour_lw,
        transform=ccrs.PlateCarree(),
        zorder=2
    )

    if label_contours:
        ax.clabel(
            contours,
            fmt="%d m",
            fontsize=8,
            inline=True,
            inline_spacing=1
        )

    # -----------------------------
    # Clip coastline polygons
    # -----------------------------

    # regional subset only, no clipping
    ice_reg = subset_dateline_gdf(ice, lat_min=lat_min, lat_max=lat_max)
    land_reg = subset_dateline_gdf(land, lat_min=lat_min, lat_max=lat_max)

    # project polygons into the map projection before plotting
    ice_proj = ice_reg.to_crs(proj.proj4_init)
    land_proj = land_reg.to_crs(proj.proj4_init)

    ax.add_geometries(
        ice_proj.geometry,
        crs=proj,
        facecolor="lightgrey",
        edgecolor="k",
        linewidth=0.3,
        zorder=3,
    )

    ax.add_geometries(
        land_proj.geometry,
        crs=proj,
        facecolor="grey",
        edgecolor="k",
        linewidth=0.3,
        zorder=4,
    )

    return fig, ax

In [ ]:
fig, ax = TNB_study_site(
    bath=bath,
    ice=ice,
    land=land,
    lon_min=162,
    lon_max=170,
    lat_min=-76,
    lat_max=-74,
    central_longitude=180,
    contour_levels=[500,800,1000,1100],
    gridline_lons=[160, 170, 180, -170, -160],
    gridline_lats=[-75, -72, -69, -66, -63, -60],
    label_contours=True,
)

stations = list(casts.keys())
lons = [float(ds.lon) for ds in casts.values()]
lats = [float(ds.lat) for ds in casts.values()]

sc = ax.scatter(
    lons,
    lats,
    c=stations,
    cmap='viridis',
    s=40,
    edgecolor='k',
    transform=ccrs.PlateCarree(),
    zorder=10
)

plt.colorbar(sc, ax=ax, label='Station')
plt.savefig('ctd_profile_TNBD_prelim/map.png')

In [ ]:
casts[40]